# This notebook was used to support the arguments presented in Chapter 2 of the MSc thesis and, in particular, to study the GRB afterglow emission observed from an off-axis point of view jointly with the prompt emission ratio on- and off-axis assuming a $\textbf{top-hat jet}$:

Importing the used packages for this notebook:

Important comment: In this notebook, the codes are not commented because of the previously commented notebooks in which an equivalent procedure was followed.

In [1]:
import numpy as np
import scipy as sc
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
import matplotlib as mpl
import redback as rd
import bilby as bb
import astropy as ast
import pandas as pd
import warnings
import extinction as ex
from astropy.table import Table, Column
from astropy import units as u
from astropy import constants as const
import astropy.io.ascii as ascii
import re
warnings.filterwarnings('ignore')



# For the interactive plot:
from ipywidgets import widgets,FloatSlider, HBox, VBox, Play, jslink, interactive_output, Output, ToggleButton, FloatRangeSlider, Layout, HTMLMath
from IPython.display import display, clear_output, Math

import logging
logging.getLogger("redback").setLevel(logging.WARNING)


def wavetof(wave):
    return(const.c.value/ (wave*1e-10))

# Function to present the results in scientific notation:
def sci_notation(x):
    exponent = int(f"{x:.0e}".split("e")[1])
    mantissa = x / 10**exponent
    return mantissa, exponent

z=0.401
ez=0.005

#Fixed GRB afterglow model parameters values
epse=np.log10(1e-1)
epsb=np.log10(1e-2)
xiN=1
thetajet=3 # In degrees

# Models that we will use to plot:
# Tophat jet model:
model='tophat_redback'



# Times over we will plot
tmin, tmax, N = -8, 8, 5000
tplot = np.logspace(tmin, tmax, N)

def updateparams(loge0,logn0,g0,thvdthc, model, expans=0, wind=0):
    parameters = {}
    parameters['redshift'] = z
    parameters['loge0'] = loge0 
    parameters['thc'] = np.deg2rad(thetajet) 
    parameters['thv'] = np.deg2rad(thvdthc*thetajet) 
    parameters['logn0']=logn0
    parameters['xiN']= xiN
    parameters['p']=2.5
    parameters['logepse']=epse
    parameters['logepsb']=epsb
    parameters['g0']=g0
    parameters['expansion']=0

    if expans==True:
        parameters['expansion']=1
    

    #Wind-like density profile condition
    if wind==True:
        parameters['k']=2

    return parameters



frequencies = np.array([5*1e9, wavetof(4671.78), wavetof(6141.12), wavetof(7457.89), wavetof(8933.78), 2.418*1e17]) #One in radio (~5GHz), one in X-rays (1keV ~ 2.418*10^17 Hz) and for in optical (g r i z)
model_kwargs = {'frequency':frequencies, 'output_format':'flux_density'}
num_data_pts= N


def simulated_data(tsim, loge0,logn0,g0,thvdthc,modelused, expans=0, wind=0):
    afterglow_simulation = rd.simulate_transients.SimulateGenericTransient(
        model=modelused,
        parameters=updateparams(loge0,logn0,g0,thvdthc, modelused, expans, wind),
        times=tsim,
        data_points=num_data_pts,
        model_kwargs=model_kwargs,
        multiwavelength_transient=True)
    return afterglow_simulation

#Function to put latex names to the widgets:
def labeled(widget, latex):
    out = widgets.Output(layout=Layout(width="120px"))
    with out:
        display(Math(latex))
    return HBox([out, widget])

# widgets
sEk = FloatSlider(value=50, min=48, max=52, step=0.1) 
sn0 = FloatSlider(value=0, min=-2, max=2, step=0.01) 
sg0 = FloatSlider(value=500, min=1.0, max=1000, step=1) 
sthv = FloatSlider(value=thetajet, min=0, max=10, step=0.01) 
winbut = ToggleButton(value=False,description='Wind-like CSM',disabled=False,button_style='warning', icon='check', tooltip='If on, activates wind-like circumstellar material mode for simulations (k=2), otherwise n_0 will be constant')
sideexp = ToggleButton(value=False,description='Lateral Expansion',disabled=False,button_style='warning', icon='check', tooltip='If on, activates lateral jet expansion for simulations, the jet will not expand sideways')
FluxLims= FloatRangeSlider(value=[-10, 0], min=-20, max=3, step=1, readout=True,readout_format='.1f')
TimeLims= FloatRangeSlider(value=[-4, tmax], min=tmin, max=tmax, step=1, readout=True,readout_format='.1f')



def make_afterglow_from_sim(sim):
    time = sim.data['time'].values
    flux = sim.data['output'].values
    freq = sim.data['frequency'].values
    ferr = sim.data['output_error'].values
    ag = rd.transient.Afterglow(name='aftersim',
                                    time=time, frequency=freq,
                                    flux_density=flux, flux_density_err=ferr,
                                    data_mode='flux_density')
    return ag

/tmp/ipykernel_4077431/1331584114.py:2: UserWarning: A NumPy version >=1.22.4 and <2.3.0 is required for this version of SciPy (detected version 2.3.5)
  import scipy as sc
/opt/anaconda3/envs/redback/lib/python3.11/site-packages/lalsimulation/lalsimulation.py:8: UserWarning: Wswiglal-redir-stdio:

SWIGLAL standard output/error redirection is enabled in IPython.
This may lead to performance penalties. To disable locally, use:

with lal.no_swig_redirect_standard_output_error():
    ...

To disable globally, use:

lal.swig_redirect_standard_output_error(False)

Note however that this will likely lead to error messages from
LAL functions being either misdirected or lost when called from
Jupyter notebooks.

To suppress this warning, use:

import warnings
warnings.filterwarnings("ignore", "Wswiglal-redir-stdio")
import lal

  import lal
18:38 bilby INFO    : Running bilby version: 2.7.1
18:38 redback INFO    : Running redback version: 1.15.3


In [2]:
def gammatobeta(gamma):
    return np.sqrt(1 - (gamma**(-2)))

def coschi(thetav,theta,phi):
    return np.cos(thetav)*np.cos(theta) +np.sin(thetav)*np.sin(theta)*np.cos(phi)

def Doppler(gamma,thetav, theta,phi):
    return 1/(gamma*(1-gammatobeta(gamma)*coschi(thetav,theta,phi)))

Ng=300 
Np=250 
Nint=200


GammaVal=np.linspace(1,1000,Ng)
VVAL=np.linspace(0,thetajet*10,Np) #In degrees
VVALM, GammaVM=np.meshgrid(VVAL, GammaVal)

phiint=np.linspace(0,2*np.pi,Nint) # In radians

Eoff=np.zeros((Ng,Np)) 
Eon=np.zeros((Ng)) 

thetaInt=np.linspace(0,np.radians(thetajet),Nint) #In radians but with thetajet in degrees

Thetagrid, Phigrid = np.meshgrid(thetaInt, phiint, indexing='ij') #Shape of this (Nint, Nint)

sinint=np.sin(Thetagrid) #Shape (Nint, Nint)

for GammaInd, Gamma in enumerate(GammaVal):
    
    VarrCalc=VVAL[:, None, None] #Shape of this: (Np,1,1)
    ThetagridCalc=Thetagrid[None,:,:] #Shape of this: (1,Nint,Nint)
    PhigridCalc=Phigrid[None,:,:] #Shape of this: (1,Nint,Nint)
    
    Doff=Doppler(Gamma,np.radians(VarrCalc),ThetagridCalc,PhigridCalc) #Shape of this: (Np,Nint,Nint)

    
    integrandoff=sinint[None,:,:]*(Doff**3) #Shape of this: (Np, Nint,Nint)
        
    
    Eofftheta=sc.integrate.simps(integrandoff,thetaInt, axis=1) #Shape after this: (Np, Nint)
    EoffVval=sc.integrate.simps(Eofftheta,phiint, axis=1) #Shape after this: (Np)
    Eoff[GammaInd,:]=EoffVval


    Don=Doppler(Gamma,0,Thetagrid,Phigrid) #Shape (Nint,Nint)
    integrandon=sinint*(Don**3) #Shape=(Nint,Nint)

    #So, integrating:
    Eontheta=sc.integrate.simps(integrandon,thetaInt,axis=0)
    EonScalar=sc.integrate.simps(Eontheta, phiint, axis=0) 
    Eon[GammaInd]=EonScalar #Shape of this (Ng)

    Eongrid=Eon[:, None] #Shape: (Ng,1)

    #Calculating the ratio:
    EisoRat=np.log10(Eoff/Eongrid)
    EisoRat_masked = np.ma.masked_invalid(EisoRat)
    #Levels of the contour I will mark:
    levels_to_mark = [-0.3,-0.5,-1,-2,-3,-4,-5]
    levels_to_mark=sorted(levels_to_mark)

# Next, in order to know the value of the ratio for a particular combination of gammas and viewing angles, 
# we can interpolate using scipy as:
EisoRatInterp=sc.interpolate.RegularGridInterpolator((np.log10(GammaVal), VVAL), EisoRat, bounds_error=True, fill_value=None) #np.log10 since I used logspace (for a smoother interpolation)

In [ ]:
# The parameter that will stablish a connection between the prompt emission and the 
# afterglow emission will be the radiative efficiency, defined as: E_prompt_iso/(E_prompt_iso + E_K_iso):
# Slider for the radiative efficiency:
effEner=FloatSlider(value=20, min=0.1, max=100, step=0.1)

# In this definition of the radiative efficiency, the kinetic energy is related to the isotropic-equivalent energy from the prompt
# emission. This is, what portion of the total kinetic energy of the jet is radiated during the prompt emission.
def EisofromEff(Eff,Ekin):
    return Eff*Ekin/(1-Eff)
    


# Creating the interactive plot:
%matplotlib inline


outap = Output()
controlsap = VBox([
    HBox([labeled(sEk, r"$\log_{10} (E_{k, 0}/\mathrm{erg})$"), 
          labeled(sn0, r"$\log_{10} (n_0/\mathrm{cm^{-3}})$"), labeled(FluxLims, r"$\log_{10} \mathrm{Flux \, Limits}$")]), 
          HBox([labeled(sg0, r"$\Gamma_0$"), 
                labeled(sthv, r"$\theta_v/\theta_j$"), labeled(TimeLims, r"$\log_{10} \mathrm{Time \, Limits}$")]), 
                HBox([labeled(effEner, r"$\eta (\%)$"), winbut, sideexp])])
display(controlsap, outap)


def update_plotap(change=None):
    loge0 = sEk.value
    logn0 = sn0.value
    g0 = sg0.value
    thvdthc = sthv.value
    winact=winbut.value
    sideexpact=sideexp.value
    effi=(effEner.value)/100
    ylimin, ylimax = FluxLims.value
    xlimin, xlimax=TimeLims.value
    with outap:
        clear_output(wait=True)
        fig2, (ax1,ax2) = plt.subplots(1, 2, figsize=(12, 4))

        sim = simulated_data(tplot, loge0, logn0, g0, thvdthc, model, sideexpact, winact)

        ag = make_afterglow_from_sim(sim)

        # Plotting with REDBACK
        ag.plot_data(axes=ax1, show=False)


        ax1.legend(['Radio [5GHz]', 'g', 'r', 'i', 'z', 'X-rays [1keV]'])
        ax1.set_xscale('log')
        ax1.set_yscale('log')
        ax1.set_xlabel('Observer-Frame Time (d)')
        ax1.set_ylabel('Flux density (mJy)')
        ax1.set_ylim([10**ylimin, 10**ylimax])
        ax1.set_xlim([10**xlimin, 10**xlimax])
        ax1.set_title(rf'Simulated GRB afterglow emission using the model: {model} in $REDBACK$')

        #Now, for the prompt emission plot: 
        cs = ax2.contourf(VVALM, GammaVM, EisoRat, levels=60, cmap=mpl.colormaps['magma'])

        cs_lines=ax2.contour(VVALM, GammaVM, EisoRat_masked,
                     levels=levels_to_mark,
                     colors='black', linewidths=1.0, linestyles='--')

        fmt = {lvl: rf'{lvl}' for lvl in levels_to_mark}
        ax2.clabel(cs_lines, fmt=fmt, inline=True, fontsize=9)
        ax2.scatter(thetajet*thvdthc,g0, s=100, marker='*', c='cyan', zorder=3)
        ax2.set_xlabel(r'$\theta_v (^{\circ})$', fontsize=18)
        ax2.set_ylabel(r'$\Gamma_0$', fontsize=18)
        ax2.set_title(rf'Prompt off/on-axis expected emission comparison: $\theta_j = {thetajet}^\circ$')
        ax2.set_yscale('log')
        prplot=fig2.colorbar(cs)
        prplot.set_label(r'$\log_{10}(E^{iso}_{off} / E^{iso}_{on})$', fontsize=18)
        ax2.set_ylim(GammaVal.min(), GammaVal.max())
        ax2.set_xlim(VVAL.min(), VVAL.max())


        fig2.tight_layout()
        plt.show()
        plt.close(fig2)

        #For this choice of parameters the Eiso that would be measured if observed on-axis would be given by:
        EisoRatchoice=EisoRatInterp([[np.log10(g0), thetajet*thvdthc]])[0]
        EisoOffAxis=(10**EisoRatchoice)*EisofromEff(effi,10**loge0) #Notice that this is approximated since we are interpolating

        EisoOffMan, EisoOffExp= sci_notation(EisoOffAxis)
        EisoOnMan, EisoOnExp= sci_notation(EisofromEff(effi,10**loge0))

        display(Math(rf"\text{{For a radiative efficiency of }} \eta ={(effi*100):.1f}\%:"))
        display(Math(rf"\text{{The prompt isotropic-equivalent energy observed on-axis would be of }} E^{{iso}}_{{on}} = {EisoOnMan:.2f} \times 10^{{{EisoOnExp}}} \text{{ ergs}}"))
        display(Math(
            rf"\text{{The prompt isotropic-equivalent energy with an off-axis ratio of }} "
            rf"\theta_v/\theta_j = {thvdthc} "
            rf"\text{{ would be of }} E^{{iso}}_{{off}} = {EisoOffMan:.2f} \times 10^{{{EisoOffExp}}} \text{{ ergs}}"
        ))

for s in (sEk, sn0, sg0, sthv, winbut, sideexp, FluxLims, TimeLims, effEner):
    s.observe(update_plotap, names='value')


update_plotap()

Output()